In [1]:
import rapidsegment as rs

In [2]:
print(f"RapidSegment version: {rs.__version__}")

RapidSegment version: 1.1.8


In [3]:
#LOAD DATA
from rapidsegment import UniversalDataLoader

In [4]:
data = UniversalDataLoader(file_path=r"/workspaces/RapidSegment/bank-full.csv", ).load()
print(f"Loaded as {type(data)} table for better performance ")
data.to_pandas().head()

2026-08-09 15:52:28,623 | INFO     | [data_loader.py:147] | 📂 Loading file: /workspaces/RapidSegment/bank-full.csv (extension: .csv)


Loaded as <class 'pyarrow.lib.Table'> table for better performance 


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,Target
0,58.0,management,married,tertiary,no,2143.0,yes,no,unknown,5.0,may,261.0,1.0,-1.0,0.0,unknown,no
1,44.0,technician,single,secondary,no,29.0,yes,no,unknown,5.0,may,151.0,1.0,-1.0,0.0,unknown,no
2,33.0,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5.0,may,76.0,1.0,-1.0,0.0,unknown,no
3,47.0,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5.0,may,92.0,1.0,-1.0,0.0,unknown,no
4,33.0,unknown,single,unknown,no,1.0,no,no,unknown,5.0,may,198.0,1.0,-1.0,0.0,unknown,no


In [5]:
#Encoding Target Variable into Binary Format
import duckdb
conn = duckdb.connect(database=':memory:', read_only=False)
conn.register("df", data)
conn.execute("CREATE OR REPLACE TABLE mod_df AS (SELECT *EXCLUDE(Target), CASE WHEN Target = 'yes' THEN 1 ELSE 0 END AS Target FROM df)")
conn.execute("SELECT * FROM mod_df").fetchdf().head()
mod_data = conn.execute("SELECT * FROM mod_df").to_arrow_table()

In [6]:
from rapidsegment import StrategicSegmentBuilder

In [7]:
help(StrategicSegmentBuilder)

Help on class StrategicSegmentBuilder in module rapidsegment.builder:

class StrategicSegmentBuilder(builtins.object)
 |  StrategicSegmentBuilder(target: str, n_jobs: int = -1, min_sample_size: int = 1000, min_lift: float = 2.0, min_events: int = 5, top_n_vars: int = 20, max_segments: int = 10, max_feature_reuse: int = 1, param_grid: Optional[Dict[str, List[Any]]] = None, enable_diversity: bool = False, enable_1way: bool = True, enable_2way: bool = True, enable_3way: bool = True, feature_groups: Optional[Dict[str, List[str]]] = None, ignore_features: Optional[List[str]] = None, sort_priority: str = 'lift_rate_count', binning_method: str = 'optimal', naive_bins: int = 5, max_expansion_hops: int = 1, selection_metric: str = 'iv', expand_log_mode: str = 'summary', db_path: Optional[str] = None, db_temp_dir: Optional[str] = None) -> None
 |
 |  Extracts hierarchical, predictive segments from tabular data.
 |
 |  The extraction is sequential:
 |      - At each step, the best rule (by lift a

In [8]:
param_grid = {'min_sample_size': [20000, 15000, 10000, 5000, 500],'min_lift': [3.0, 2.0, 1.5]}
builder = StrategicSegmentBuilder(target = 'Target',
                                  min_sample_size = 100,
                                  min_lift = 1.5, 
                                  min_events  = 50, 
                                  top_n_vars = 10,
                                  max_segments = 10,
                                  param_grid = param_grid,
                                  enable_diversity = False, 
                                  max_feature_reuse = 5, 
                                  enable_1way = True,
                                  enable_2way = True,
                                  enable_3way = False,
                                  feature_groups = None,
                                  ignore_features = None,
                                  sort_priority = 'events_rate_lift',
                                  binning_method = 'naive',
                                  naive_bins =20,
                                  selection_metric = 'response_rate',
                                  expand_log_mode = 'full', 
                                  max_expansion_hops = 5,)

2026-08-09 15:52:30,299 | INFO     | [builder.py:157] | 📂 Created disk-backed DB at: experiments/segmentation_20260809_046890b9.duckdb


In [9]:
segments_df = builder.extract_segments(mod_data)

2026-08-09 15:52:30,631 | INFO     | [builder.py:838] | 🚀 Starting hierarchical segment extraction...
2026-08-09 15:52:30,905 | INFO     | [builder.py:852] | ⚙️ DuckDB Configured for Disk Spilling: Threads=4/4, MemoryLimit=12GB, TempDir=experiments/tmp_20260809_046890b9
2026-08-09 15:52:30,906 | INFO     | [builder.py:856] | 📊 Sort priority: events_rate_lift
2026-08-09 15:52:30,907 | INFO     | [builder.py:857] | 📦 Binning method: naive (naive_bins=20)
2026-08-09 15:52:31,266 | INFO     | [builder.py:881] | 📊 Dynamic Grid Search Enabled: 15 configurations.
2026-08-09 15:52:31,268 | INFO     | [builder.py:892] | 🔒 Locking Original Base Rate: 11.70%
2026-08-09 15:52:31,270 | INFO     | [builder.py:909] | 🔄 Iteration 1 | Remaining Volume: 45,211 | Base Rate: 11.70%
2026-08-09 15:52:31,271 | INFO     | [builder.py:256] | 🔍 Computing IV and bins for 16 features...
2026-08-09 15:52:31,993 | INFO     | [builder.py:678] | 🔀 Adjacent-bin expansion summary
2026-08-09 15:52:31,994 | INFO     | [b

In [10]:
final_eval = builder.evaluate_final_coverage(mod_data)

2026-08-09 15:52:36,045 | INFO     | [builder.py:1244] | 📊 Evaluating final hierarchical coverage on original data...


In [11]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(segments_df).columns)
for _, row in pd.DataFrame(segments_df).iterrows():
    table.add_row(list(row))
print(table)

+------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------+-------+--------------------+--------------------+--------------------------+-----------------------+
| segment_id |                                                      rule_string                                                       |                sql_filter                | count |        rate        |        lift        | meta_applied_sample_size | meta_applied_min_lift |
+------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------+-------+--------------------+--------------------+--------------------------+-----------------------+
|     1      |        duration=[[280.0, 319.0), [319.0, 368.0), [368.0, 437.0), [437.0, 548.0), [548.0, 751.0), [751.0, inf)]         |           (duration >= 2

In [12]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------------+---------------------+---------------------------+--------------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |         lift        | cumulative_sample_capture | cumulative_event_capture |
+---------+-------------+---------------+--------------------+--------------------+--------------------+---------------------+---------------------------+--------------------------+
|   1.0   |   13570.0   |     3619.0    | 26.66912306558585  | 11.698480458295547 | 30.014819402357833 |  2.279708305763286  |     30.014819402357833    |    68.42503308754017     |
|   2.0   |    4120.0   |     775.0     | 18.810679611650485 | 11.698480458295547 |  9.11282652451837  |  1.6079592284407829 |     39.127645926876205    |    83.07808659481944     |
|   3.0   |    550.0    |     236.0     | 42.90909090909091  | 11.698480458295547 | 1.2165

In [13]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   duration=[[280.0, 319.0), [319.0, 368.0), [368.0, 437.0), [437.0, 548.0), [548.0, 751.0), [751.0, inf)]
SQL Filter: (duration >= 280.0)
--------------------------------------------------
Segment ID: 2
Raw Rule:   duration=[[176.0, 190.0), [190.0, 205.0), [205.0, 220.0), [220.0, 238.0), [238.0, 257.0), [257.0, inf)] & housing=[no]
SQL Filter: (duration >= 176.0) AND (housing = 'no')
--------------------------------------------------
Segment ID: 3
Raw Rule:   poutcome=[success]
SQL Filter: (poutcome = 'success')
--------------------------------------------------


In [14]:
builder.explain_feature_journey("campaign")

📌 AUDIT TRAIL FOR FEATURE: 'campaign'

[Iteration 1]
  • Current dynamic RESPONSE_RATE   : 0.1460
  • Previous times used  : 0
  • Selection Status     : Excluded (Outside Top N Features by Score)
  • Winner this round    : duration=[[280.0, 319.0), [319.0, 368.0), [368.0, 437.0), [437.0, 548.0), [548.0, 751.0), [751.0, inf)] (Variables: ['duration'])

[Iteration 2]
  • Current dynamic RESPONSE_RATE   : 0.0782
  • Previous times used  : 0
  • Selection Status     : Excluded (Outside Top N Features by Score)
  • Winner this round    : duration=[[176.0, 190.0), [190.0, 205.0), [205.0, 220.0), [220.0, 238.0), [238.0, 257.0), [257.0, inf)] & housing=[no] (Variables: ['duration', 'housing'])

[Iteration 3]
  • Current dynamic RESPONSE_RATE   : 0.0491
  • Previous times used  : 0
  • Selection Status     : Excluded (Outside Top N Features by Score)
  • Winner this round    : poutcome=[success] (Variables: ['poutcome'])

[Iteration 4]
  • Current dynamic RESPONSE_RATE   : 0.0368
  • Previous 

In [15]:
# Preparing the dataset for scoring and decile banding.
conn.register("predicted", mod_data)
predicted = conn.query("""
                        SELECT *, 
                        CASE WHEN duration >= 706.50 AND job IN ('management', 'unemployed')
                        THEN 1 ELSE 0 END AS seg_1,
                        CASE WHEN duration >= 647.50 AND job IN ('technician')
                        THEN 1 ELSE 0 END AS seg_2,
                        CASE WHEN duration >= 632.50 AND job IN ('unknown', 'self-employed', 'admin.', 'unemployed')
                        THEN 1 ELSE 0 END AS seg_3,
                        CASE WHEN (pdays >= 8.50 AND pdays < 200.50) AND poutcome IN ('other', 'success')
                        THEN 1 ELSE 0 END AS seg_4,
                        CASE WHEN month IN ('apr', 'sep', 'oct', 'dec', 'mar') AND age >= 58.50
                        THEN 1 ELSE 0 END AS seg_5,
                        CASE WHEN duration >= 472.50
                        THEN 1 ELSE 0 END AS seg_6,
                        CASE WHEN month IN ('apr', 'sep', 'dec', 'oct', 'mar')
                        THEN 1 ELSE 0 END AS seg_7,                                                                                        
                        ROW_NUMBER() OVER () AS ID,
                        FROM predicted
""").df()
conn.close()

In [16]:
#Score the segments on the dataset and create decile bands
from rapidsegment import StrategicSegmentScore
scorer = StrategicSegmentScore(
    target_col="Target",
    primary_key="ID",
    segment_cols=["seg_1","seg_2",'seg_3','seg_4','seg_5','seg_6','seg_7'],
)

In [17]:
model_artifact = scorer.calculate_and_export_weights(predicted)

2026-08-09 15:52:37,835 | INFO     | [scorer.py:71] | 🚀 Initialising out‑of‑core DuckDB scorecard engine...
2026-08-09 15:52:38,425 | INFO     | [scorer.py:113] | 📊 Computing scorecard weights...
2026-08-09 15:52:38,426 | WARNING  | [scorer.py:157] | ⚠️ DECILE RESOLUTION WARNING: Only 7 distinct non-zero score values found across 7 segments. Splitting into 10 deciles will produce repeated thresholds (e.g., top 5 deciles may have identical scores). For smooth decile ranking, ensure the builder discovers at least 10 distinct segments (increase `max_segments`). Consider interpreting results as tiers rather than deciles.
2026-08-09 15:52:38,427 | INFO     | [scorer.py:171] | ⚡ Scoring population natively via SQL engine...
2026-08-09 15:52:38,552 | INFO     | [scorer.py:191] | 📉 Dataset Zero‑Inflation Rate: 88.30%
2026-08-09 15:52:38,553 | INFO     | [scorer.py:196] | 📈 Calibrating deciles across active populations...
2026-08-09 15:52:38,558 | INFO     | [scorer.py:244] | ✅ Scorecard export

In [18]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 54
Segment: seg_2 | Weight: 51
Segment: seg_3 | Weight: 52
Segment: seg_4 | Weight: 50
Segment: seg_5 | Weight: 43
Segment: seg_6 | Weight: 41
Segment: seg_7 | Weight: 31


In [19]:
model_artifact.get("decile_min_thresholds")

{'1': 228,
 '2': 93,
 '3': 81,
 '4': 50,
 '5': 50,
 '6': 41,
 '7': 41,
 '8': 41,
 '9': 31,
 '10': 31}

In [20]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 54 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 51 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 52 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 50 ELSE 0 END AS seg_4_weighted,
    CASE WHEN seg_5 = 1 THEN 43 ELSE 0 END AS seg_5_weighted,
    CASE WHEN seg_6 = 1 THEN 41 ELSE 0 END AS seg_6_weighted,
    CASE WHEN seg_7 = 1 THEN 31 ELSE 0 END AS seg_7_weighted
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted + seg_5_weighted + seg_6_weighted + seg_7_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=228 THEN 1
                    WHEN total_weight >= 93 THEN 2
                    WHEN total_weight >= 81 THEN 3
                    WHEN total_weight >= 50 THEN 4
                    WHEN total_weight >= 50 THEN 5
                    WHEN total_weight >= 41 THEN 6
                    WHEN total_weight >= 41 THEN 7
                    WHEN total_weight >= 41 THEN 8
                    WHEN total_weight >= 31 THEN 9
                    WHEN total_weight >= 31 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [21]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(Target) AS events, 
                    (SUM(Target)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,0,33944,1248.0,3.676644
1,1,3,3.0,100.000000
2,2,1480,813.0,54.932432
3,3,1015,561.0,55.270936
4,4,2012,802.0,39.860835
5,6,3544,1184.0,33.408578
6,9,3213,678.0,21.101774
